# Graph closure analysis: Does the separation-one overlay add useful output?

Compare `D` (filtered direct Is a edges) with `D ∪ P` (`P`: minimum-separation-one ancestry pairs). Athena root depths and target eligibility stay fixed. This notebook starts with invented input; the optional final cell requires an explicit Athena path. It does not modify the vocabulary or library defaults.

In [ ]:
from pathlib import Path
import sys
import tempfile

repo = Path.cwd() if (Path.cwd() / 'pyhealth').is_dir() else Path.cwd().parent
assert (repo / 'pyhealth').is_dir(), 'Start Jupyter in the repository or examples directory'
sys.path.insert(0, str(repo))
from examples.keep_closure_mode_audit import write_synthetic_bundle, run_audit
from pyhealth.medcode import OMOPVocabulary
from pyhealth.tasks import build_keep_omop_ancestor_rollup

temporary = tempfile.TemporaryDirectory()
folder = Path(temporary.name) / 'synthetic'
write_synthetic_bundle(folder)
summary = run_audit(folder, Path(temporary.name) / 'audit', synthetic=True)
summary['graph_counts']

## Synthetic expectations

Twelve Condition concepts are in U, with six eligible targets and five deep sources. Two added edges have different effects: `104 → 201` recovers an orphan; `102 → 204` adds a first-retained target already reachable under all-retained. A filtered Observation between `104` and `202` needs a longer shortcut, so separation-one overlay cannot repair that mapping.

These tables intentionally stress authoritative depths, including disagreement with adjacency distances. Their hand-authored ancestry rows are a semantic fixture, not a simulated complete Athena release.

In [ ]:
assert summary['graph_counts'] == dict(U=12, D=9, P=11, intersection=9, P_minus_D=2, D_minus_P=0, union=11)
for policy, result in summary['policies'].items():
    population = result['populations']['all']
    assert population['denominator'] == 12
    assert population['direct_no_target'] == 3
    assert population['closure_no_target'] == 2
    assert population['categories']['recovered_no_target'] == 1
    print(policy, population['categories'])

## Branch stopping, graph gaps, and failures

Both policies preserve old targets when acyclic edges are only added at fixed eligibility. An existing path to a first-retained target still exists with no newly eligible internal node.

A disconnected eligible source (`203`) maps to itself; a deep source (`202`) stays `no_target`. The excluded root is also `no_target` and is counted separately from deep failures. Diagnostic enumeration uses `on_unmapped="empty"`; the production default is `"error"`.

In [ ]:
vocabulary = OMOPVocabulary(folder)
for mode in ('direct', 'closure'):
    processor = build_keep_omop_ancestor_rollup(vocabulary, target_policy='first_retained', include_root=False, path_source=mode, on_unmapped='empty')
    rows = processor.rollup_codes(['201', '202', '203', '204'])
    print(mode, [(r.source, r.targets, r.reason) for r in rows])
    assert rows[1].reason == 'no_target'
    assert rows[2].targets == ('203',)
    eligible = build_keep_omop_ancestor_rollup(vocabulary, target_policy='first_retained', include_root=False, path_source=mode, max_depth=7, on_unmapped='empty')
    assert eligible.rollup_codes(['202'])[0].targets == ('202',)
assert summary['policies']['first_retained']['populations']['all']['categories']['changed_nonempty_targets'] == 2
assert summary['policies']['all_retained']['populations']['all']['categories']['changed_nonempty_targets'] == 0

## Explicit opt-in release audit

Set `ATHENA_DIR` to an authorized local export and choose `LOCAL_RESULTS` outside shared artifacts. No path discovery or automatic fallback occurs. The helper evaluates every concept once per mode and policy in batches. It records release metadata, table hashes, failures, differences and small path witnesses. Full ancestor tables are materialized by the public reference reader and this baseline factory; this is included in resource reporting.

Release-specific rows remain local. Vocabulary behavior does not establish diagnosis prevalence, clinical benefit, embedding quality, or fidelity to unpublished author artifacts.

In [ ]:
ATHENA_DIR = None  # Explicit Path('/your/authorized/athena-export') to opt in.
LOCAL_RESULTS = repo / '.artifacts' / 'graph-closure-analysis-athena'
if ATHENA_DIR is not None:
    real_summary = run_audit(Path(ATHENA_DIR), LOCAL_RESULTS, synthetic=False)
    print(real_summary['graph_counts'])
else:
    print('Athena execution skipped; synthetic results only in this execution.')

In [ ]:
temporary.cleanup()